In [5]:
# Ensure required packages are available (run this cell once)
%pip install requests pandas openpyxl pyproj shapely

# Imports (placed here so the installation cell also ensures modules are importable in this notebook)
import requests
import pandas as pd
from pyproj import Transformer
from shapely.geometry import LineString, box
from shapely.strtree import STRtree


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: C:\Users\Caleb\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Railway

In [62]:
r"""
Hospital-centered railway exposure (1914) from MartinBrake KMLs.

"""

import os
import time
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from pyproj import Transformer
from shapely.geometry import LineString, box
from shapely.strtree import STRtree

try:
    import numpy as np
except Exception:
    np = None


# ---------------------------
# SETTINGS
# ---------------------------
HOSPITAL_FILE = r"C:\Users\Caleb\Downloads\scraping\adress_19.xlsx"

BASE_URL = "https://www.martinbrake.de/timelines/data/bahn/"
OUTPUT_DIR = "bahn_kml"

YEAR = 1914
CELL_SIZE_M = 50_000
HALF = CELL_SIZE_M / 2

OUT_DIR = os.path.dirname(HOSPITAL_FILE)
OUT_CSV = os.path.join(OUT_DIR, f"2hospitals_rail_length_{YEAR}_{CELL_SIZE_M}m.csv")
OUT_XLSX = os.path.join(OUT_DIR, f"2hospitals_rail_length_{YEAR}_{CELL_SIZE_M}m.xlsx")

STORE_PIECES = False
OUT_PIECES_CSV = os.path.join(OUT_DIR, f"hospitals_rail_pieces_{YEAR}_{CELL_SIZE_M}m.csv")

RAW_TYPES = [
    {"id": "NE",  "gl": [1, 2]},
    {"id": "ND",  "gl": [1, 2]},
    {"id": "HE",  "gl": [1, 2]},
    {"id": "HD",  "gl": [1, 2]},
    {"id": "K",   "gl": [1]},
    {"id": "HGV", "gl": [2]},
]

COUNTRIES = [
    {"id": "NRW"}, {"id": "HE"}, {"id": "NI"}, {"id": "SH"}, {"id": "RPL"},
    {"id": "BW"}, {"id": "BY"}, {"id": "TH"}, {"id": "SNA"}, {"id": "SN"},
    {"id": "BR"}, {"id": "MP"},
]


# ---------------------------
# HELPERS
# ---------------------------
def to_year(s):
    if not s:
        return None
    try:
        return int(str(s).split("-")[0])
    except Exception:
        return None


def generate_file_ids():
    ids = []
    for country in COUNTRIES:
        for rtype in RAW_TYPES:
            for gl in rtype["gl"]:
                ids.append(f"{country['id']}_{rtype['id']}{gl}")
    return ids


def download_kml(file_id, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    url = f"{BASE_URL}{file_id}.kml"
    headers = {
        "Referer": "https://www.martinbrake.de/timelines/bahn",
        "User-Agent": "Mozilla/5.0",
    }
    r = requests.get(url, headers=headers, timeout=60)
    r.raise_for_status()
    path = os.path.join(out_dir, f"{file_id}.kml")
    with open(path, "wb") as f:
        f.write(r.content)
    return path


def load_hospitals(path: str) -> pd.DataFrame:
    df = pd.read_excel(path)

    required = ["KID", "KName", "GPS_l", "GPS_b"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in hospital file: {missing}")

    df = df.rename(columns={"GPS_l": "lon", "GPS_b": "lat"})
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df = df.dropna(subset=["lon", "lat"]).copy()

    df["hospital_id"] = df["KID"]

    keep = ["hospital_id", "KName", "lon", "lat"]
    for extra in ["Str", "PLZ", "Ort", "Bundesland", "Adresse", "KV"]:
        if extra in df.columns:
            keep.append(extra)

    return df[keep].copy()


def parse_linestring_coords(coords_text: str):
    pts = []
    for token in coords_text.strip().split():
        parts = token.split(",")
        if len(parts) >= 2:
            try:
                pts.append((float(parts[0]), float(parts[1])))
            except ValueError:
                continue
    return pts


def candidates_to_indices(candidates, poly_to_idx=None):
    """
    Shapely 2.x: candidates are indices (ints)
    Shapely 1.x: candidates are geometry objects
    Return list[int] hospital indices.
    """
    if candidates is None:
        return []

    try:
        if len(candidates) == 0:
            return []
    except TypeError:
        return []

    first = candidates[0]

    if isinstance(first, (int,)) or (np is not None and isinstance(first, (np.integer,))):
        return [int(i) for i in candidates]

    if poly_to_idx is None:
        raise ValueError("poly_to_idx mapping required when STRtree returns geometries.")

    return [poly_to_idx[id(geom)] for geom in candidates]


# ---------------------------
# MAIN
# ---------------------------
def main():
    wgs_to_m = Transformer.from_crs("EPSG:4326", "EPSG:3035", always_xy=True)

    hospitals = load_hospitals(HOSPITAL_FILE)
    print(f"Hospitals loaded: {len(hospitals):,}")

    hx, hy = wgs_to_m.transform(hospitals["lon"].to_numpy(), hospitals["lat"].to_numpy())
    hospitals["x_center_m"] = hx
    hospitals["y_center_m"] = hy

    cell_polys = [
        box(x - HALF, y - HALF, x + HALF, y + HALF)
        for x, y in zip(hospitals["x_center_m"], hospitals["y_center_m"])
    ]

    tree = STRtree(cell_polys)
    poly_to_idx = {id(p): i for i, p in enumerate(cell_polys)}

    total_len_m = [0.0] * len(hospitals)
    hit_count = [0] * len(hospitals)
    pieces_rows = []

    kml_paths = []
    for fid in generate_file_ids():
        path = os.path.join(OUTPUT_DIR, f"{fid}.kml")
        if not os.path.exists(path):
            try:
                print(f"Downloading {fid} ...")
                path = download_kml(fid, OUTPUT_DIR)
                time.sleep(0.2)
            except Exception as ex:
                print(f"Warning: failed {fid}: {ex}")
                continue
        kml_paths.append(path)

    print(f"KML files ready: {len(kml_paths)}")

    checked_lines = 0
    active_lines = 0
    hit_any_cell = 0

    for kml_path in kml_paths:
        root = ET.parse(kml_path).getroot()

        for placemark in root.findall(".//Placemark"):
            coords_el = placemark.find(".//LineString/coordinates")
            if coords_el is None or not coords_el.text:
                continue
            checked_lines += 1

            timespan = placemark.find(".//TimeSpan")
            if timespan is None:
                continue

            begin = timespan.find("begin").text if timespan.find("begin") is not None else None
            end = timespan.find("end").text if timespan.find("end") is not None else None

            begin_year = to_year(begin)
            end_year = to_year(end)

            if begin_year is None or begin_year > YEAR:
                continue
            if end_year is not None and end_year < YEAR:
                continue

            active_lines += 1

            ll_pts = parse_linestring_coords(coords_el.text)
            if len(ll_pts) < 2:
                continue

            lons = [p[0] for p in ll_pts]
            lats = [p[1] for p in ll_pts]
            xs, ys = wgs_to_m.transform(lons, lats)
            line = LineString(list(zip(xs, ys)))
            if line.is_empty or line.length <= 0:
                continue

            candidates = tree.query(line)
            cand_idx = candidates_to_indices(candidates, poly_to_idx=poly_to_idx)
            if len(cand_idx) == 0:
                continue

            hit_any_cell += 1

            for hidx in cand_idx:
                poly = cell_polys[hidx]
                inter = line.intersection(poly)
                if inter.is_empty:
                    continue
                seg_len = inter.length
                if seg_len <= 0:
                    continue

                total_len_m[hidx] += float(seg_len)
                hit_count[hidx] += 1

                if STORE_PIECES:
                    pieces_rows.append({
                        "hospital_id": hospitals.iloc[hidx]["hospital_id"],
                        "source_kml": os.path.basename(kml_path),
                        "line_begin_year": begin_year,
                        "segment_length_m": float(seg_len),
                    })

    out = hospitals.copy()
    out["cell_size_m"] = CELL_SIZE_M
    out[f"rail_length_in_cell_km_{YEAR}"] = [m / 1000.0 for m in total_len_m]
    out[f"rail_segments_hitting_cell_{YEAR}"] = hit_count

    out["x_min"] = out["x_center_m"] - HALF
    out["x_max"] = out["x_center_m"] + HALF
    out["y_min"] = out["y_center_m"] - HALF
    out["y_max"] = out["y_center_m"] + HALF

    out.to_csv(OUT_CSV, index=False)

    with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
        out.to_excel(writer, sheet_name=f"hospital_cells_{YEAR}", index=False)
        meta = pd.DataFrame([{
            "hospital_file": HOSPITAL_FILE,
            "year_snapshot": YEAR,
            "cell_size_m": CELL_SIZE_M,
            "kml_base_url": BASE_URL,
            "kml_files_used": len(kml_paths),
            "placemarks_with_linestring": checked_lines,
            "active_lines_in_year": active_lines,
            "active_lines_hitting_any_hospital_cell": hit_any_cell,
        }])
        meta.to_excel(writer, sheet_name="meta", index=False)

    if STORE_PIECES:
        pd.DataFrame(pieces_rows).to_csv(OUT_PIECES_CSV, index=False)

    print("DONE")
    print("Wrote:", OUT_CSV)
    print("Wrote:", OUT_XLSX)


if __name__ == "__main__":
    main()


Hospitals loaded: 2,293
KML files ready: 120
DONE
Wrote: C:\Users\Caleb\Downloads\scraping\2hospitals_rail_length_1914_50000m.csv
Wrote: C:\Users\Caleb\Downloads\scraping\2hospitals_rail_length_1914_50000m.xlsx


# AUTOBAHN

In [63]:
r"""
AUTOBAHN exposure pipeline — FULL CODE with GEOMETRY DEDUPLICATION.

Dedup strategy implemented:
1) For each LineString, compute a stable hash of its coordinates after:
   - rounding coordinates to a metric grid (tolerance in meters)
   - direction-invariant ordering (A->B equals B->A)
2) Keep only one geometry per hash.

This eliminates exact/near-exact duplicates across KML files.

Requirements:
  pip install requests pandas openpyxl pyproj shapely
"""

import os
import re
import json
import time
import hashlib
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from pyproj import Transformer
from shapely.geometry import LineString, box
from shapely.strtree import STRtree

try:
    import numpy as np
except Exception:
    np = None


# =========================================================
# SETTINGS
# =========================================================
HOSPITAL_FILE = r"C:\Users\Caleb\Downloads\scraping\adress_19.xlsx"

URLS_TXT = r"C:\Users\Caleb\Downloads\scraping\autobahn_urls.txt"
TIMELINE_REFERER = "https://www.martinbrake.de/timelines/autobahn"

YEAR = 1939
CELL_SIZE_M = 50_000
HALF = CELL_SIZE_M / 2

# IMPORTANT: Autobahn data often lacks TimeSpan per feature.
INCLUDE_IF_NO_TIME = True

# ---- DEDUP SETTINGS ----
# Snap coordinates to a grid of this size (meters) before hashing.
# 1.0–5.0 is typical. If you still see inflation, increase slightly (e.g. 2.0 or 5.0).
DEDUP_TOL_M = 1.0

OUT_DIR = os.path.dirname(HOSPITAL_FILE)
DOWNLOAD_DIR = os.path.join(OUT_DIR, "autobahn_downloads")

OUT_CSV = os.path.join(OUT_DIR, f"2hospitals_autobahn_length_{YEAR}_{CELL_SIZE_M}m.csv")
OUT_XLSX = os.path.join(OUT_DIR, f"2hospitals_autobahn_length_{YEAR}_{CELL_SIZE_M}m.xlsx")

STORE_PIECES = False
OUT_PIECES_CSV = os.path.join(OUT_DIR, f"2hospitals_autobahn_pieces_{YEAR}_{CELL_SIZE_M}m.csv")


# =========================================================
# HELPERS
# =========================================================
def to_year_any(val):
    if val is None:
        return None

    if isinstance(val, (int, float)) and not isinstance(val, bool):
        if abs(val) > 10_000_000_000:  # unix ms
            try:
                import datetime
                return datetime.datetime.utcfromtimestamp(val / 1000.0).year
            except Exception:
                return None
        if 1000 <= int(val) <= 3000:
            return int(val)
        return None

    s = str(val).strip()
    if not s:
        return None

    m = re.search(r"(\d{4})", s)
    if m:
        try:
            y = int(m.group(1))
            if 0 < y <= 3000:
                return y
        except Exception:
            pass
    return None


def line_active_in_year(begin_year, end_year, year, include_if_no_time=True):
    if begin_year is None and end_year is None:
        return bool(include_if_no_time)
    if begin_year is None:
        return end_year is None or end_year >= year
    if begin_year > year:
        return False
    if end_year is not None and end_year < year:
        return False
    return True


def load_hospitals(path: str) -> pd.DataFrame:
    df = pd.read_excel(path)

    required = ["KID", "KName", "GPS_l", "GPS_b"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.rename(columns={"GPS_l": "lon", "GPS_b": "lat"})
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df = df.dropna(subset=["lon", "lat"]).copy()

    df["hospital_id"] = df["KID"]

    keep = ["hospital_id", "KName", "lon", "lat"]
    for extra in ["Str", "PLZ", "Ort", "Bundesland", "Adresse", "KV"]:
        if extra in df.columns:
            keep.append(extra)

    return df[keep].copy()


def candidates_to_indices(candidates, poly_to_idx=None):
    if candidates is None:
        return []
    try:
        if len(candidates) == 0:
            return []
    except TypeError:
        return []

    first = candidates[0]
    if isinstance(first, (int,)) or (np is not None and isinstance(first, np.integer)):
        return [int(i) for i in candidates]

    if poly_to_idx is None:
        raise ValueError("poly_to_idx required when STRtree returns geometries")
    return [poly_to_idx[id(g)] for g in candidates]


def read_urls_list(path: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing URL list file: {path}")
    urls = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            urls.append(s)
    return urls


def download_url(url: str, out_dir: str) -> str:
    os.makedirs(out_dir, exist_ok=True)
    name = os.path.basename(url.split("?")[0]) or "data"
    local = os.path.join(out_dir, name)

    if os.path.exists(local) and os.path.getsize(local) > 0:
        return local

    headers = {"Referer": TIMELINE_REFERER, "User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers, timeout=120)
    r.raise_for_status()

    with open(local, "wb") as f:
        f.write(r.content)

    return local


# =========================================================
# PARSERS
# =========================================================
def _kml_text(el):
    return el.text.strip() if (el is not None and el.text) else ""


def _parse_kml_linestring_coordinates(coords_text: str):
    pts = []
    for token in coords_text.strip().split():
        parts = token.split(",")
        if len(parts) >= 2:
            try:
                pts.append((float(parts[0]), float(parts[1])))
            except Exception:
                continue
    return pts


def _parse_gx_track_coords(placemark):
    pts = []
    for c in placemark.findall(".//{*}coord"):
        txt = _kml_text(c)
        if not txt:
            continue
        parts = txt.split()
        if len(parts) >= 2:
            try:
                pts.append((float(parts[0]), float(parts[1])))
            except Exception:
                continue
    return pts


def _extract_kml_feature_years(placemark):
    ts = placemark.find(".//{*}TimeSpan")
    if ts is not None:
        by = to_year_any(_kml_text(ts.find(".//{*}begin")))
        ey = to_year_any(_kml_text(ts.find(".//{*}end")))
        return by, ey

    props = {}
    for data in placemark.findall(".//{*}ExtendedData//{*}Data"):
        name = data.get("name") or data.get("{name}")
        val = _kml_text(data.find(".//{*}value"))
        if name:
            props[str(name).lower()] = val

    for a, b in [("begin", "end"), ("start", "stop"), ("from", "to"), ("open", "close")]:
        if a in props or b in props:
            return to_year_any(props.get(a)), to_year_any(props.get(b))

    return None, None


def parse_kml_lines(file_path: str, year: int, wgs_to_m: Transformer, include_if_no_time=True):
    lines = []
    root = ET.parse(file_path).getroot()

    for placemark in root.findall(".//{*}Placemark"):
        begin_year, end_year = _extract_kml_feature_years(placemark)
        if not line_active_in_year(begin_year, end_year, year, include_if_no_time=include_if_no_time):
            continue

        found_any = False
        for ls in placemark.findall(".//{*}LineString"):
            coords_el = ls.find(".//{*}coordinates")
            coords_text = _kml_text(coords_el)
            if not coords_text:
                continue
            ll_pts = _parse_kml_linestring_coordinates(coords_text)
            if len(ll_pts) < 2:
                continue
            xs, ys = wgs_to_m.transform([p[0] for p in ll_pts], [p[1] for p in ll_pts])
            geom = LineString(list(zip(xs, ys)))
            if not geom.is_empty and geom.length > 0:
                lines.append(geom)
                found_any = True

        if not found_any:
            ll_pts = _parse_gx_track_coords(placemark)
            if len(ll_pts) >= 2:
                xs, ys = wgs_to_m.transform([p[0] for p in ll_pts], [p[1] for p in ll_pts])
                geom = LineString(list(zip(xs, ys)))
                if not geom.is_empty and geom.length > 0:
                    lines.append(geom)

    return lines


def parse_geojson_lines(file_path: str, year: int, wgs_to_m: Transformer, include_if_no_time=True):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        txt = f.read().strip()

    try:
        data = json.loads(txt)
    except Exception:
        return []

    feats = None
    if isinstance(data, dict) and data.get("type") == "FeatureCollection":
        feats = data.get("features", [])
    elif isinstance(data, dict) and data.get("type") == "Feature":
        feats = [data]
    else:
        return []

    def get_years(props):
        if not isinstance(props, dict):
            return (None, None)

        pairs = [
            ("begin", "end"),
            ("start", "stop"),
            ("from", "to"),
            ("open", "close"),
            ("opening_year", "closing_year"),
            ("built", "removed"),
        ]
        low = {str(k).lower(): props[k] for k in props.keys()}

        for a, b in pairs:
            if a in low or b in low:
                return to_year_any(low.get(a)), to_year_any(low.get(b))

        for k in ["year", "opened", "opening", "start_year"]:
            if k in low:
                return to_year_any(low.get(k)), None

        return (None, None)

    lines = []
    for feat in feats:
        if not isinstance(feat, dict):
            continue
        geom = feat.get("geometry")
        props = feat.get("properties", {}) or {}
        if not isinstance(geom, dict):
            continue

        by, ey = get_years(props)
        if not line_active_in_year(by, ey, year, include_if_no_time=include_if_no_time):
            continue

        gtype = geom.get("type")
        coords = geom.get("coordinates")

        if gtype == "LineString" and isinstance(coords, list) and len(coords) >= 2:
            pts = [(c[0], c[1]) for c in coords if isinstance(c, list) and len(c) >= 2]
            if len(pts) < 2:
                continue
            xs, ys = wgs_to_m.transform([p[0] for p in pts], [p[1] for p in pts])
            ls = LineString(list(zip(xs, ys)))
            if not ls.is_empty and ls.length > 0:
                lines.append(ls)

        elif gtype == "MultiLineString" and isinstance(coords, list):
            for part in coords:
                if not isinstance(part, list) or len(part) < 2:
                    continue
                pts = [(c[0], c[1]) for c in part if isinstance(c, list) and len(c) >= 2]
                if len(pts) < 2:
                    continue
                xs, ys = wgs_to_m.transform([p[0] for p in pts], [p[1] for p in pts])
                ls = LineString(list(zip(xs, ys)))
                if not ls.is_empty and ls.length > 0:
                    lines.append(ls)

    return lines


def load_all_lines_metric(files, year, wgs_to_m, include_if_no_time=True):
    all_lines = []
    for fp in files:
        low = fp.lower()
        try:
            if low.endswith(".kml"):
                all_lines.extend(parse_kml_lines(fp, year, wgs_to_m, include_if_no_time=include_if_no_time))
            elif low.endswith(".geojson") or low.endswith(".json"):
                all_lines.extend(parse_geojson_lines(fp, year, wgs_to_m, include_if_no_time=include_if_no_time))
        except Exception:
            continue
    return all_lines


# =========================================================
# DEDUP
# =========================================================
def _round_to_grid(v: float, tol: float) -> int:
    return int(round(v / tol))


def _linestring_hash_metric(ls: LineString, tol_m: float) -> str:
    """
    Direction-invariant hash of a LineString's metric coordinates.
    Rounds coords to a tol_m grid to absorb tiny numeric differences.
    """
    coords = list(ls.coords)
    if len(coords) < 2:
        return ""

    # quantize
    q = [(_round_to_grid(x, tol_m), _round_to_grid(y, tol_m)) for x, y in coords]

    # direction-invariant: choose lexicographically smaller of forward/backward
    fwd = q
    rev = list(reversed(q))
    canon = fwd if fwd <= rev else rev

    b = repr(canon).encode("utf-8")
    return hashlib.sha1(b).hexdigest()


def deduplicate_lines(lines, tol_m: float):
    seen = set()
    unique = []
    for ls in lines:
        try:
            h = _linestring_hash_metric(ls, tol_m)
        except Exception:
            continue
        if not h:
            continue
        if h in seen:
            continue
        seen.add(h)
        unique.append(ls)
    return unique, len(seen)


# =========================================================
# MAIN
# =========================================================
def main():
    wgs_to_m = Transformer.from_crs("EPSG:4326", "EPSG:3035", always_xy=True)

    hospitals = load_hospitals(HOSPITAL_FILE)
    print(f"Hospitals loaded: {len(hospitals):,}")

    hx, hy = wgs_to_m.transform(hospitals["lon"].to_numpy(), hospitals["lat"].to_numpy())
    hospitals["x_center_m"] = hx
    hospitals["y_center_m"] = hy

    cell_polys = [
        box(x - HALF, y - HALF, x + HALF, y + HALF)
        for x, y in zip(hospitals["x_center_m"], hospitals["y_center_m"])
    ]

    tree = STRtree(cell_polys)
    poly_to_idx = {id(p): i for i, p in enumerate(cell_polys)}

    total_len_m = [0.0] * len(hospitals)
    hit_count = [0] * len(hospitals)
    pieces_rows = []

    urls = read_urls_list(URLS_TXT)
    downloaded = []
    for u in urls:
        try:
            downloaded.append(download_url(u, DOWNLOAD_DIR))
            time.sleep(0.05)
        except Exception as ex:
            print(f"Warning: failed download {u}: {ex}")

    if not downloaded:
        raise RuntimeError("No Autobahn data files downloaded. Fix autobahn_urls.txt URLs.")

    print(f"Data files ready: {len(downloaded)}")

    lines = load_all_lines_metric(downloaded, YEAR, wgs_to_m, include_if_no_time=INCLUDE_IF_NO_TIME)
    if not lines:
        sizes = [(os.path.basename(p), os.path.getsize(p)) for p in downloaded]
        raise RuntimeError(
            "No line geometries parsed. "
            f"Downloaded files (name,size): {sizes}. "
            "Likely causes: URLs are not geometry files, or data is not LineString/MultiLineString, "
            "or it is vector tiles (pbf/mvt) not supported by this script."
        )

    print(f"Polylines parsed (raw): {len(lines):,}")

    lines_unique, _ = deduplicate_lines(lines, DEDUP_TOL_M)
    print(f"Polylines after dedup (@{DEDUP_TOL_M}m): {len(lines_unique):,}")

    hit_any_cell = 0

    for line in lines_unique:
        candidates = tree.query(line)
        cand_idx = candidates_to_indices(candidates, poly_to_idx=poly_to_idx)
        if not cand_idx:
            continue

        hit_any_cell += 1

        for hidx in cand_idx:
            poly = cell_polys[hidx]
            inter = line.intersection(poly)
            if inter.is_empty:
                continue
            seg_len = inter.length
            if seg_len <= 0:
                continue

            total_len_m[hidx] += float(seg_len)
            hit_count[hidx] += 1

            if STORE_PIECES:
                pieces_rows.append({
                    "hospital_id": hospitals.iloc[hidx]["hospital_id"],
                    "segment_length_m": float(seg_len),
                })

    out = hospitals.copy()
    out["cell_size_m"] = CELL_SIZE_M
    out[f"autobahn_length_in_cell_km_{YEAR}"] = [m / 1000.0 for m in total_len_m]
    out[f"autobahn_segments_hitting_cell_{YEAR}"] = hit_count

    out["x_min"] = out["x_center_m"] - HALF
    out["x_max"] = out["x_center_m"] + HALF
    out["y_min"] = out["y_center_m"] - HALF
    out["y_max"] = out["y_center_m"] + HALF

    out.to_csv(OUT_CSV, index=False)

    with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
        out.to_excel(writer, sheet_name=f"hospital_cells_{YEAR}", index=False)
        meta = pd.DataFrame([{
            "hospital_file": HOSPITAL_FILE,
            "url_list_file": URLS_TXT,
            "timeline_referer": TIMELINE_REFERER,
            "year_snapshot": YEAR,
            "cell_size_m": CELL_SIZE_M,
            "include_if_no_time": INCLUDE_IF_NO_TIME,
            "dedup_tol_m": DEDUP_TOL_M,
            "downloaded_files": len(downloaded),
            "parsed_lines_raw": len(lines),
            "parsed_lines_dedup": len(lines_unique),
            "lines_hitting_any_cell": hit_any_cell,
        }])
        meta.to_excel(writer, sheet_name="meta", index=False)

    if STORE_PIECES and pieces_rows:
        pd.DataFrame(pieces_rows).to_csv(OUT_PIECES_CSV, index=False)

    print("DONE")
    print("Wrote:", OUT_CSV)
    print("Wrote:", OUT_XLSX)


if __name__ == "__main__":
    main()


Hospitals loaded: 2,293
Data files ready: 5
Polylines parsed (raw): 234
Polylines after dedup (@1.0m): 122
DONE
Wrote: C:\Users\Caleb\Downloads\scraping\2hospitals_autobahn_length_1939_50000m.csv
Wrote: C:\Users\Caleb\Downloads\scraping\2hospitals_autobahn_length_1939_50000m.xlsx


# Waterways

In [64]:
"""
Waterway exposure around hospitals (VerkNet-BWaStr dataset)

What this version does:
- Downloads VerkNet-BWaStr ZIP
- Reads polylines
  - Prefers pyshp if installed (best attribute reliability)
  - Falls back to geopandas/pyogrio and STILL enforces exclusions by a detected name column
- Reprojects everything to EPSG:3035
- Builds 10km x 10km squares around hospitals
- Clips lines to each square, sums intersected lengths (km)
- Adds your manual LINESTRINGs (straight-line abstractions) on top of the dataset lines
- Removes the listed waterways from the dataset before exposure calculation

Dependencies (one of the readers is required):
  Preferred: pip install pyshp
  Fallback:  pip install geopandas pyogrio

Minimum core:
  pip install requests pyproj shapely openpyxl pandas
"""

import os
import zipfile
import requests

import pandas as pd
from pyproj import Transformer, CRS
from shapely.geometry import LineString, box
from shapely.strtree import STRtree

try:
    import numpy as np
except ImportError:
    np = None


# ----------------------------------------------------------------------
# SETTINGS
HOSPITAL_FILE = r"C:\Users\Caleb\Downloads\scraping\adress_19.xlsx"

WATERWAYS_ZIP_URL = (
    "https://www.gdws.wsv.bund.de/SharedDocs/Downloads/DE/Karten/VerkNet-BWaStr.zip"
    "?__blob=publicationFile"
)

YEAR = 1939  # only used for naming outputs (dataset has no time)
CELL_SIZE_M = 50_000
HALF = CELL_SIZE_M / 2

OUT_DIR = os.path.dirname(HOSPITAL_FILE)
OUT_CSV = os.path.join(OUT_DIR, f"2hospitals_waterways_length_{YEAR}_{CELL_SIZE_M}m.csv")
OUT_XLSX = os.path.join(OUT_DIR, f"2hospitals_waterways_length_{YEAR}_{CELL_SIZE_M}m.xlsx")

STORE_PIECES = False
OUT_PIECES_CSV = os.path.join(OUT_DIR, f"2hospitals_waterways_pieces_{YEAR}_{CELL_SIZE_M}m.csv")

# ----------------------------------------------------------------------
# EXCLUSIONS (exact strings as provided)
EXCLUDE_WATERWAYS_RAW = [
    "Altmühl, Altmühl",
    "Elbe-Seitenkanal, Elbe-Seitenkanal",
    "Havelkanal, Havelkanal",
    "Main-Donau-Kanal, Altarm bei Pautzfeld",
    "Main-Donau-Kanal, Altarm Kastlhof",
    "Main-Donau-Kanal, Altkanal Ludwig-Donau-Main-Kanal",
    "Main-Donau-Kanal, Altstrecke Alter Werkkanal",
    "Main-Donau-Kanal, Hauptstrecke",
    "Main-Donau-Kanal, Kraftwerkskanal Hirschaid",
    "Main-Donau-Kanal, Pumpwerkskanal Neuessing",
    "Main-Donau-Kanal, Wehrarm Forchheim",
    "Main-Donau-Kanal, Wehrstrecke Kelheim",
    "Main-Donau-Kanal, Wehrstrecke Riedenburg",
    "Mosel, Altarm Nehren",
    "Mosel, Altarm Taubengrün",
    "Mosel, Altarm Winningen",
    "Mosel, Hauptstrecke",
    "Mosel, Nebenarm Hatzenport",
    "Mosel, Nebenarm Lehmen",
    "Mosel, Nebenarm Pfalzel-Ehrang",
    "Mosel, Nebenarm Treis",
    "Mosel, Nebenarm Trier",
    "Mosel, Nebenarm Zell",
    "Mosel, Nebenarm Ürzig",
    "Mosel, Wehrarm Detzem",
    "Mosel, Wehrstrecke Enkirch",
    "Mosel, Wehrstrecke Fankel",
    "Mosel, Wehrstrecke Grevenmacher",
    "Mosel, Wehrstrecke Koblenz",
    "Mosel, Wehrstrecke Lehmen",
    "Mosel, Wehrstrecke Müden",
    "Mosel, Wehrstrecke Palzem",
    "Mosel, Wehrstrecke St. Aldegund",
    "Mosel, Wehrstrecke Trier",
    "Mosel, Wehrstrecke Wintrich",
    "Mosel, Wehrstrecke Zeltingen",
    "Regnitz, Flussstrecke linker Regnitzarm Bamberg",
    "Regnitz, Hauptstrecke",
    "Saar, Altarm Ensdorf",
    "Saar, Altarm Saarbrücken Stadenanlage",
    "Saar, Altarm Schwemlingen",
    "Saar, Altarm Wadgassen",
    "Saar, Flussstrecke Nied (Mündungsstrecke) mit Altarm",
    "Saar, Km 94,06 (= lothr. km 75,62) bis 0,00",
    "Saar, lothr. km 64,98 re bis 75,62 li (Grenzstrecke mit Frankreich)",
    "Saar, Wehrarm Kleinblittersdorf",
    "Saar, Wehrarm Rilchingen-Hanweiler",
    "Saar, Wehrarm Schoden (Wiltinger Bogen)",
    "Saar, Wehrstrecke Güdingen",
    "Saar, Wehrstrecke Lisdorf",
    "Saar, Wehrstrecke Mettlach",
    "Saar, Wehrstrecke Rehlingen",
    "Saar, Wehrstrecke Saarbrücken",
    "Saar, Wehrstrecke Serrig",
]

def _norm_name(s: str) -> str:
    if s is None:
        return ""
    return " ".join(str(s).strip().split()).casefold()

EXCLUDE_SET = {_norm_name(s) for s in EXCLUDE_WATERWAYS_RAW}


# ----------------------------------------------------------------------
# MANUAL EXTRA LINES (WGS84 lon/lat) -> converted to EPSG:3035
MANUAL_LINES_WGS84 = [
    ("Rhein–Marne-Kanal", [(7.7600617, 48.5496317), (6.7064850, 48.6908117)]),
    ("Saar Canal", [(6.9058650, 48.7048533), (7.0639750, 49.1097533)]),
    ("Hamme-Oste Canal", [(9.0238600, 53.4289833), (9.1636667, 53.4966100)]),
    ("Nordhorn–Almelo-Kanal", [(6.6387033, 52.3647733), (7.0787867, 52.4331950)]),
    ("Coevorden–Piccardie Canal", [(6.7091150, 52.6276367), (7.0550217, 52.6440200)]),
    ("Ruppiner Kanal", [(12.8112550, 52.9213500), (13.2323933, 52.7700217)]),
    ("Breitenburg Canal", [(9.5504600, 53.9068283), (9.5999183, 53.8820650)]),
    ("South-North Channel", [(7.1205200, 52.8390400), (7.0780217, 52.4342117)]),
]


# ----------------------------------------------------------------------
# HELPERS
def download_zip(url: str, out_path: str) -> str:
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        return out_path
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers, timeout=180)
    r.raise_for_status()
    with open(out_path, "wb") as f:
        f.write(r.content)
    return out_path


def extract_zip(zip_path: str, out_dir: str) -> str:
    base = os.path.splitext(os.path.basename(zip_path))[0]
    target_dir = os.path.join(out_dir, base)
    os.makedirs(target_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(target_dir)
    return target_dir


def load_hospitals(path: str) -> pd.DataFrame:
    df = pd.read_excel(path)
    req = ["KID", "KName", "GPS_l", "GPS_b"]
    missing = [c for c in req if c not in df.columns]
    if missing:
        raise ValueError(f"Missing hospital columns: {missing}")

    df = df.rename(columns={"GPS_l": "lon", "GPS_b": "lat"})
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df = df.dropna(subset=["lon", "lat"]).copy()
    df["hospital_id"] = df["KID"]

    keep = ["hospital_id", "KName", "lon", "lat"]
    for extra in ["Str", "PLZ", "Ort", "Bundesland", "Adresse", "KV"]:
        if extra in df.columns:
            keep.append(extra)
    return df[keep].copy()


def build_poly_wkb_index(polys):
    return {polys[i].wkb: i for i in range(len(polys))}


def candidates_to_indices(candidates, wkb_to_idx):
    if candidates is None:
        return []
    try:
        if len(candidates) == 0:
            return []
    except TypeError:
        return []
    first = candidates[0]
    if isinstance(first, int) or (np is not None and isinstance(first, np.integer)):
        return [int(i) for i in candidates]
    return [wkb_to_idx[g.wkb] for g in candidates]


def _pick_shp_file(shp_dir: str) -> str:
    shp_files = [f for f in os.listdir(shp_dir) if f.lower().endswith(".shp")]
    if not shp_files:
        for root, _, files in os.walk(shp_dir):
            for f in files:
                if f.lower().endswith(".shp"):
                    return os.path.join(root, f)
        raise FileNotFoundError(f"No .shp found under: {shp_dir}")
    return os.path.join(shp_dir, shp_files[0])


def _read_source_crs(shp_path: str) -> CRS:
    prj_path = os.path.splitext(shp_path)[0] + ".prj"
    if os.path.exists(prj_path):
        with open(prj_path, "r", encoding="utf-8", errors="ignore") as f:
            wkt = f.read()
        try:
            return CRS.from_wkt(wkt)
        except Exception:
            pass
    return CRS.from_epsg(25832)


def _detect_name_field(field_names):
    candidates = []
    for fn in field_names:
        s = (fn or "").strip()
        if not s:
            continue
        low = s.casefold()
        if "name" in low and "filename" not in low:
            candidates.append(s)
    for pref in ["NAME", "Name", "name"]:
        if pref in field_names:
            return pref
    if candidates:
        return candidates[0]
    return None


def load_waterway_lines_with_names(shp_dir: str):
    """
    Yields (LineString in EPSG:3035, name_or_None)

    Priority:
    1) pyshp (attribute-aware → exclusions applied)
    2) geopandas (attribute-aware if possible → exclusions applied)
    """
    shp_path = _pick_shp_file(shp_dir)

    # ---------- Reader 1: pyshp ----------
    try:
        import shapefile  # pyshp

        sf = shapefile.Reader(shp_path)
        field_names = [f[0] for f in sf.fields[1:]]
        print("FIELDS IN SHAPEFILE:", field_names)

        name_field = _detect_name_field(field_names)
        if name_field is None:
            raise RuntimeError("Cannot apply exclusions: no name-like field found (pyshp).")

        name_idx = field_names.index(name_field)

        source_crs = _read_source_crs(shp_path)
        to_3035 = Transformer.from_crs(source_crs, 3035, always_xy=True)

        removed = 0
        kept = 0

        for shapeRec in sf.iterShapeRecords():
            shp = shapeRec.shape
            if shp.shapeType not in (
                shapefile.POLYLINE,
                shapefile.POLYLINEZ,
                shapefile.POLYLINEM,
            ):
                continue

            rec = shapeRec.record
            name = rec[name_idx] if rec and len(rec) > name_idx else None
            if _norm_name(name) in EXCLUDE_SET:
                removed += 1
                continue

            for i, start in enumerate(shp.parts):
                end = shp.parts[i + 1] if i + 1 < len(shp.parts) else len(shp.points)
                pts = shp.points[start:end]
                if len(pts) < 2:
                    continue
                xs, ys = to_3035.transform([p[0] for p in pts], [p[1] for p in pts])
                ls = LineString(zip(xs, ys))
                if ls.length > 0:
                    kept += 1
                    yield ls, name

        print(f"Applied exclusions via pyshp field '{name_field}': removed {removed} features.")
        return

    except ModuleNotFoundError:
        print("pyshp not installed → trying geopandas attribute-based exclusions.")

    # ---------- Reader 2: geopandas fallback ----------
    try:
        import geopandas as gpd
    except ModuleNotFoundError as e:
        raise ModuleNotFoundError(
            "Need a shapefile reader. Install ONE:\n"
            "  pip install pyshp\n"
            "  pip install geopandas pyogrio\n"
        ) from e

    gdf = gpd.read_file(shp_path)
    if gdf.crs is None:
        gdf = gdf.set_crs(_read_source_crs(shp_path))
    gdf = gdf.to_crs(epsg=3035)

    cols = [c for c in gdf.columns if c != "geometry"]
    name_col = None
    for c in cols:
        lc = str(c).casefold()
        if "name" in lc and "filename" not in lc:
            name_col = c
            break

    if name_col is None:
        raise RuntimeError(
            "Cannot apply exclusions: geopandas loaded the shapefile but no name-like column was found."
        )

    gdf["_nm_norm"] = gdf[name_col].apply(_norm_name)
    before = len(gdf)
    gdf = gdf[~gdf["_nm_norm"].isin(EXCLUDE_SET)].copy()
    after = len(gdf)
    print(f"Applied exclusions via geopandas column '{name_col}': removed {before - after} features.")

    for geom, nm in zip(gdf.geometry, gdf[name_col]):
        if geom is None or geom.is_empty:
            continue
        if geom.geom_type == "LineString":
            if geom.length > 0:
                yield geom, nm
        elif geom.geom_type == "MultiLineString":
            for part in geom.geoms:
                if part.length > 0:
                    yield part, nm


def manual_lines_as_3035():
    wgs_to_3035 = Transformer.from_crs(4326, 3035, always_xy=True)
    for nm, pts in MANUAL_LINES_WGS84:
        lons = [p[0] for p in pts]
        lats = [p[1] for p in pts]
        xs, ys = wgs_to_3035.transform(lons, lats)
        ls = LineString(list(zip(xs, ys)))
        if ls.length > 0:
            yield ls, nm


# ----------------------------------------------------------------------
# MAIN
def main():
    # 1) Load hospitals, project to EPSG:3035, build squares
    hospitals = load_hospitals(HOSPITAL_FILE)
    wgs_to_3035 = Transformer.from_crs(4326, 3035, always_xy=True)
    hx, hy = wgs_to_3035.transform(
        hospitals["lon"].to_numpy(),
        hospitals["lat"].to_numpy(),
    )
    hospitals["x_center_m"] = hx
    hospitals["y_center_m"] = hy

    cell_polys = [
        box(x - HALF, y - HALF, x + HALF, y + HALF)
        for x, y in zip(hospitals["x_center_m"], hospitals["y_center_m"])
    ]
    tree = STRtree(cell_polys)
    wkb_to_idx = build_poly_wkb_index(cell_polys)

    # 2) Download and extract waterways dataset
    zip_local = os.path.join(OUT_DIR, "VerkNet-BWaStr.zip")
    download_zip(WATERWAYS_ZIP_URL, zip_local)
    shp_dir = extract_zip(zip_local, OUT_DIR)

    # 3) Load waterway lines (EPSG:3035) with enforced exclusions + add manual lines
    lines_with_names = list(load_waterway_lines_with_names(shp_dir))
    manual_with_names = list(manual_lines_as_3035())

    if not lines_with_names:
        raise RuntimeError("No line geometries found in the waterways dataset (empty after parsing).")

    all_lines = lines_with_names + manual_with_names

    # 4) Clip lines to hospital squares, sum lengths
    total_len_m = [0.0] * len(hospitals)
    hit_count = [0] * len(hospitals)
    pieces_rows = []

    for (ls, nm) in all_lines:
        candidates = tree.query(ls)
        cand_idx = candidates_to_indices(candidates, wkb_to_idx)
        if not cand_idx:
            continue

        for hidx in cand_idx:
            inter = ls.intersection(cell_polys[hidx])
            if inter.is_empty:
                continue

            seg_len = 0.0
            if inter.geom_type == "LineString":
                seg_len = inter.length
            elif inter.geom_type == "MultiLineString":
                seg_len = sum(g.length for g in inter.geoms)
            else:
                continue

            if seg_len <= 0:
                continue

            total_len_m[hidx] += float(seg_len)
            hit_count[hidx] += 1

            if STORE_PIECES:
                pieces_rows.append(
                    {
                        "hospital_id": hospitals.iloc[hidx]["hospital_id"],
                        "waterway_name": nm,
                        "segment_length_m": float(seg_len),
                    }
                )

    # 5) Write outputs
    out = hospitals.copy()
    out["cell_size_m"] = CELL_SIZE_M
    out[f"waterway_length_in_cell_km_{YEAR}"] = [m / 1000.0 for m in total_len_m]
    out[f"waterway_segments_hitting_cell_{YEAR}"] = hit_count

    out["x_min"] = out["x_center_m"] - HALF
    out["x_max"] = out["x_center_m"] + HALF
    out["y_min"] = out["y_center_m"] - HALF
    out["y_max"] = out["y_center_m"] + HALF

    out.to_csv(OUT_CSV, index=False)

    with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
        out.to_excel(writer, sheet_name=f"hospital_cells_{YEAR}", index=False)
        meta = pd.DataFrame(
            [
                {
                    "hospital_file": HOSPITAL_FILE,
                    "waterway_zip_url": WATERWAYS_ZIP_URL,
                    "year_snapshot": YEAR,
                    "cell_size_m": CELL_SIZE_M,
                    "downloaded_zip": zip_local,
                    "extracted_dir": shp_dir,
                    "dataset_polylines_loaded_after_exclusion": len(lines_with_names),
                    "manual_polylines_added": len(manual_with_names),
                    "total_polylines_used": len(all_lines),
                    "excluded_names_count": len(EXCLUDE_SET),
                }
            ]
        )
        meta.to_excel(writer, sheet_name="meta", index=False)

    if STORE_PIECES and pieces_rows:
        pd.DataFrame(pieces_rows).to_csv(OUT_PIECES_CSV, index=False)

    print("DONE")
    print("Wrote:", OUT_CSV)
    print("Wrote:", OUT_XLSX)


if __name__ == "__main__":
    main()


pyshp not installed → trying geopandas attribute-based exclusions.


C:\Users\Caleb\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\pyogrio\raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured LineString' is converted to 'LineString'
  return ogr_read(


Applied exclusions via geopandas column 'NAME': removed 59 features.
DONE
Wrote: C:\Users\Caleb\Downloads\scraping\2hospitals_waterways_length_1939_50000m.csv
Wrote: C:\Users\Caleb\Downloads\scraping\2hospitals_waterways_length_1939_50000m.xlsx
